In [ ]:
'''This part contains the code for my second experiment in the paper'''

In [ ]:
def get_style_representation(j, style_dataset, loss_network, imsize=256):
    content_nodes = ['relu_3_3']
    style_nodes = ['relu_1_2', 'relu_2_2', 'relu_3_3', 'relu_4_3']
    return_nodes = {3: 'relu_1_2', 8: 'relu_2_2', 15: 'relu_3_3', 22: 'relu_4_3'}
    mean_style_gram = {}

    style_count = 0
    for layer_name in return_nodes.values():
        mean_style_gram[layer_name] = 0

    for img_path, style_id in style_dataset.images:
        if (style_id == j):
            style_img = imload(img_path, imsize=imsize)
            style_count += 1
            features = loss_network(style_img)
            for layer_name, feature in features.items():
                mean_style_gram[layer_name] += gram(feature)

    for layer_name, gram_matrix in mean_style_gram.items():
        if style_count > 0:
            mean_style_gram[layer_name] = gram_matrix / style_count
    return mean_style_gram

In [ ]:
'''Epoch-based training loop'''
save_interval = 100
start_epoch = 1
current_model_checkpoint = ""
batch_size = 16 # (or 32, 64, 128,.... powers-of-2 is the convention)
num_epochs = 3000

def train(content_weight = 1.0, style_weight=10.0, tv_weight=1e-5, lr=1e-3):
    content_nodes = ['relu_3_3']
    style_nodes = ['relu_1_2', 'relu_2_2', 'relu_3_3', 'relu_4_3']
    return_nodes = {3: 'relu_1_2',
                    8: 'relu_2_2',
                    15: 'relu_3_3',
                    22: 'relu_4_3'}

    vgg = vgg16(weights=VGG16_Weights.IMAGENET1K_V1).features
    for param in vgg.parameters():
        param.requires_grad = False
    loss_network = create_feature_extractor(vgg, return_nodes)
    loss_network = loss_network.to(device)

    ''' This part pre-calculate style gram matrices once before training loop
    style_grams_path refers to the variable that saves the path of the pre-calculated style gram matrices'''

    if not os.path.exists(style_grams_path):
      precomputed_style_grams = {}
      for style_id in range(NUM_STYLE):
        precomputed_style_grams[style_id] = get_style_representation(style_id, style_dataset, loss_network, imsize=256)
      torch.save(precomputed_style_grams, style_grams_path)
    else:
      precomputed_style_grams = torch.load(style_grams_path)

    for style_id, dictionary in precomputed_style_grams.items():
      for layer_name, gram_matrix in dictionary.items():
        gram_matrix = gram_matrix.to(device)
        precomputed_style_grams[style_id][layer_name] = gram_matrix


    # network
    model = StyleTransferNetwork()
    if current_model_checkpoint:
      model.load_state_dict(torch.load(current_model_checkpoint))

    model = model.to(device)
    model.train()
    optimizer = Adam(model.parameters(), lr=lr)
    content_iterator = iter(content_dataloader)

    print("Start training...")
    #-------------------

    # New -- Epoch Loop
    for epoch in range(start_epoch, 1 + num_epochs):

        print(f'Epoch {epoch} / {num_epochs}')


        '''
        These variables will keep track of the cumulative loss over all the batches in the content_dataset in this epoch.
        Then in the next epoch, they'll be re-initialized to 0.
        '''
        cumulative_total_loss = 0.0
        cumulative_style_loss = 0.0
        cumulative_content_loss = 0.0

        '''
        Loops through the content_dataset in batches of batch_size, and makes model-weight updates after each batch has been processed.
        Every image in the content_dataset will be seen once in this loop before you start the next epoch.
        Then in the next epoch, all content images will be covered once again, and so on....
        '''
        for batch_idx, (content_images, content_images_idxs) in enumerate(content_dataloader):
          print(f'Batch {batch_idx} / {len(content_dataloader)}')
          style_id = random.randint(0, NUM_STYLE - 1)


          current_batch_size = content_images.shape[0]
          style_codes = torch.zeros(current_batch_size, NUM_STYLE, 1)
          style_codes[:, style_id, :] = 1 # Set the corresponding style_id to 1 for all items in the batch
          style_codes = style_codes.to(device)

          content_images = content_images.to(device)

          output_images = model(content_images, style_codes)


          output_features = loss_network(output_images)
          content_features = loss_network(content_images)

          style_features_gram = precomputed_style_grams[style_id]

          style_loss = calc_style_loss_custom (output_features, style_features_gram, style_nodes)
          content_loss = calc_content_loss (output_features, content_features, content_nodes)
          tv_loss = calc_tv_loss(output_images)

          total_loss = content_loss * content_weight + style_loss * style_weight + tv_loss * tv_weight

          '''
          Keeps track of accumulating losses over all the batches in this epoch.
          In the next epoch, these values will be re-initialized to 0.
          '''
          cumulative_total_loss += total_loss.item()
          cumulative_style_loss += style_loss.item()
          cumulative_content_loss += content_loss.item()


          optimizer.zero_grad()
          total_loss.backward()
          optimizer.step()

        # This is done at the level of per epoch (not per batch!)
        total_losses.append(cumulative_total_loss / len(content_dataloader))
        style_losses.append(cumulative_style_loss /  len(content_dataloader))
        content_losses.append(cumulative_content_loss / len(content_dataloader))


        global loss_path
        save_losses(loss_path)

        # New -- [-1] index records the average loss in the current epoch
        if (epoch % save_interval == 0):
          print(f'Epoch {epoch} / {num_epochs} | Style Loss: {style_losses[-1]:.4f} | Content Loss: {content_losses[-1]:.4f}')
          path = f"/content/drive/MyDrive/Movie_Project/Tuning_style_weights/content_1_style_10000/Avg_gram/Model/Model_epoch_{epoch}.pth"
          # The path is customized to the link where I want to save the output checkpoint models.

          torch.save(model.state_dict(), path)
    return model